In [4]:
import pandas as pd

df = pd.read_csv("marketing.csv")
df

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain,Country
0,1826,1970,Graduation,Divorced,84835.0,0,0,6/16/2014,0,189,...,6,1,0,0,0,0,0,1,0,Spain
1,1,1961,Graduation,Single,57091.0,0,0,6/15/2014,0,464,...,7,5,0,0,0,0,1,1,0,Canada
2,10476,1958,Graduation,Married,67267.0,0,1,5/13/2014,0,134,...,5,2,0,0,0,0,0,0,0,USA
3,1386,1967,Graduation,Together,32474.0,1,1,5/11/2014,0,10,...,2,7,0,0,0,0,0,0,0,Australia
4,5371,1989,Graduation,Single,21474.0,1,0,4/8/2014,0,6,...,2,7,1,0,0,0,0,1,0,Spain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2235,10142,1976,PhD,Divorced,66476.0,0,1,3/7/2013,99,372,...,11,4,0,0,0,0,0,0,0,USA
2236,5263,1977,2n Cycle,Married,31056.0,1,0,1/22/2013,99,5,...,3,8,0,0,0,0,0,0,0,Spain
2237,22,1976,Graduation,Divorced,46310.0,1,0,12/3/2012,99,185,...,5,8,0,0,0,0,0,0,0,Spain
2238,528,1978,Graduation,Married,65819.0,0,0,11/29/2012,99,267,...,10,3,0,0,0,0,0,0,0,India


1. Test Continu (T-test d'Indépendance) : Le Test de Corrélation

Hypothèse Nulle (H0) : Le revenu n'a aucune influence sur les plaintes. Les deux groupes ont la même moyenne de revenus.     
Hypothèse Alternative (H1) : Le revenu influence les plaintes. Il y a un écart réel entre les revenus des plaignants et des non-plaignants

In [7]:
import scipy.stats as stats

# 1. Traitement de la data : On nettoie les revenus (suppression des NaNs et espaces)
df.columns = df.columns.str.strip()
df_clean = df.dropna(subset=['Income'])

# 2. Séparation des groupes
groupe_plainte = df_clean[df_clean['Complain'] == 1]['Income']
groupe_satisfait = df_clean[df_clean['Complain'] == 0]['Income']

# 3. Le Test
t_stat, p_val = stats.ttest_ind(groupe_plainte, groupe_satisfait)

print(f"Test Continu (Revenu vs Plainte) - P-value : {p_val:.4f}")

Test Continu (Revenu vs Plainte) - P-value : 0.2002


Vue que p-value = 0.2>0.05 , on ne peut pas rejetter h0,et bien evidemment le revenu n'est pas un coupable statistiquement significatif pour expliquer les plaintes

2. Test Discret (Chi-deux chi2) : Le Test de Dépendance

Hypothèse Métier : "Le succès de la campagne marketing dépend du statut matrimonial (ex: les célibataires sont plus réactifs aux offres que les couples).    
H0: Le statut matrimonial et la réponse à la campagne sont indépendants     
H1 : Le statut matrimonial influence significativement l'acceptation de l'offre

In [6]:
from scipy.stats import chi2_contingency

# 1. Traitement : On regarde les fréquences
tableau_croise = pd.crosstab(df['Marital_Status'], df['Response'])

# 2. Le Test
chi2, p_val, dof, expected = chi2_contingency(tableau_croise)

print(f"Test Discret (Statut vs Réponse) - P-value : {p_val:.4f}")

Test Discret (Statut vs Réponse) - P-value : 0.0000


In [7]:
# Afficher le tableau brut
print(tableau_croise)

Response          0    1
Marital_Status          
Absurd            1    1
Alone             2    1
Divorced        184   48
Married         766   98
Single          374  106
Together        520   60
Widow            58   19
YOLO              1    1


Même si les clients mariés sont les plus nombreux dans notre base, ce sont les célibataires qui sont les plus réactifs à nos campagnes marketing. 
Ils convertissent beaucoup mieux

Nous avons prouvé statistiquement que la situation familiale influence directement l'acceptation de nos offres. Nous ne devons plus envoyer le même mail à tout le monde


3. Test ANOVA : Le Test de Comparaison de Groupes
(Plusieurs catégories vs Continu)

Hypothèse Nulle (H0) : Le niveau d'éducation n'a aucun impact sur les dépenses en viande.La moyenne des dépenses est la même pour tous les groupes (Graduation, PhD, Master, etc.).                       
Hypothèse Alternative (H1) : Le niveau d'éducation influence significativement les dépenses. Au moins un des groupes dépense beaucoup plus (ou moins) que les autres


In [9]:
from scipy.stats import f_oneway

# 1. Traitement : Création d'une liste de groupes sans boucles complexes
# On groupe par éducation et on récupère les dépenses en viande
groupes = [group['MntMeatProducts'] for name, group in df.groupby('Education')]

# 2. Le Test
f_stat, p_val = f_oneway(*groupes)

print(f"Test Mixte (Éducation vs Viande) - P-value : {p_val:.4e}")

Test Mixte (Éducation vs Viande) - P-value : 1.8741e-06


Le lien est très fort. Le niveau d'études permet de prédire la valeur du panier moyen dans un rayon spécifique. C'est un excellent moyen de personnaliser les promotions de produits frais

Impact de l'Éducation sur la Consommation : Résultat Hautement Significatif